# Addresses26 — 03 Agent Attribution Pilot for BN Sales, Last 5 Years

**Purpose:** test whether estate-agent attribution is viable for BN postcode Land Registry sales without contaminating the verified Land Registry core dataset.

This notebook is deliberately conservative:

- Uses your verified `pp_bn_standard_YYYY.parquet` files as the factual base.
- Builds a small, reviewable sample from the last five years.
- Generates search/evidence tasks for each transaction.
- Supports manual evidence capture first.
- Does **not** bypass anti-bot systems, logins, paywalls, CAPTCHAs, or website restrictions.
- Stores agent evidence as a separate enrichment table with confidence scores.

Expected Drive structure:

```text
/content/drive/MyDrive/Addresses26_Data/
  derived/land_registry_price_paid/pp_bn_standard_2021.parquet ...
  reports/agent_attribution/
  evidence/agent_attribution/
```


In [12]:
# === 1. Mount Drive and imports ===
from google.colab import drive

drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import json
import re
import time
import hashlib
import urllib.parse

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

BASE = Path('/content/drive/MyDrive/Addresses26_Data')
DERIVED = BASE / 'derived' / 'land_registry_price_paid'
REPORTS = BASE / 'reports' / 'agent_attribution'
EVIDENCE = BASE / 'evidence' / 'agent_attribution'

REPORTS.mkdir(parents=True, exist_ok=True)
EVIDENCE.mkdir(parents=True, exist_ok=True)

RUN_UTC = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
print('Run UTC:', RUN_UTC)
print('DERIVED:', DERIVED)
print('REPORTS:', REPORTS)
print('EVIDENCE:', EVIDENCE)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Run UTC: 20260427T132744Z
DERIVED: /content/drive/MyDrive/Addresses26_Data/derived/land_registry_price_paid
REPORTS: /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution
EVIDENCE: /content/drive/MyDrive/Addresses26_Data/evidence/agent_attribution


In [13]:
# === 2. Configuration ===
# Last 5 complete/recent years. Adjust if needed.
YEARS = [2021, 2022, 2023, 2024, 2025]
POSTCODE_PREFIX = 'BN'
STANDARD_ONLY = True

# Keep pilot deliberately small at first.
# Increase once the manual/evidence loop is working.
SAMPLE_N = 250
RANDOM_SEED = 26

# Optional targeting: set to e.g. ['BN1', 'BN2', 'BN7'] or leave empty for all BN districts.
TARGET_DISTRICTS = []

# We generate search tasks; we do not scrape portals by default.
GENERATE_SEARCH_URLS = True

print('Years:', YEARS)
print('Sample size:', SAMPLE_N)


Years: [2021, 2022, 2023, 2024, 2025]
Sample size: 250


In [14]:
# === 3. Load last 5 years of verified BN Land Registry parquet ===
frames = []
missing_files = []

for year in YEARS:
    path = DERIVED / f'pp_bn_standard_{year}.parquet'
    if not path.exists():
        missing_files.append(str(path))
        continue
    df = pd.read_parquet(path)
    df['source_year_file'] = year
    frames.append(df)

if missing_files:
    print('Missing files:')
    for p in missing_files:
        print(' -', p)

assert frames, 'No parquet files loaded. Check DERIVED path and year files.'

sales = pd.concat(frames, ignore_index=True)
print('Loaded rows:', len(sales))
print('Columns:', list(sales.columns))
sales.head()


Loaded rows: 68685
Columns: ['transaction_id', 'price', 'date_of_transfer', 'postcode', 'property_type', 'old_new', 'duration', 'paon', 'saon', 'street', 'locality', 'town_city', 'district', 'county', 'ppd_category_type', 'record_status', 'postcode_area', 'postcode_district', 'postcode_sector', 'property_type_label', 'old_new_label', 'duration_label', 'address_compact', 'source_year_file']


,transaction_id,price,date_of_transfer,postcode,property_type,old_new,duration,paon,saon,street,locality,town_city,district,county,ppd_category_type,record_status,postcode_area,postcode_district,postcode_sector,property_type_label,old_new_label,duration_label,address_compact,source_year_file
0,{F87E72F8-D9D5-176C-E053-6B04A8C0D2BE},650000,2021-01-03,BN3 5DL,S,N,F,58,<NA>,PORTLAND ROAD,<NA>,HOVE,BRIGHTON AND HOVE,BRIGHTON AND HOVE,A,A,BN,<NA>,None,Semi-detached,Established,Freehold,"58, PORTLAND ROAD, HOVE",2021
1,{BC8936BB-74DB-0E2C-E053-6C04A8C0DBF4},265000,2021-01-04,BN1 3RT,F,N,L,1,SECOND FLOOR FLAT,WEST HILL ROAD,<NA>,BRIGHTON,BRIGHTON AND HOVE,BRIGHTON AND HOVE,A,A,BN,<NA>,None,Flat/Maisonette,Established,Leasehold,"SECOND FLOOR FLAT, 1, WEST HILL ROAD, BRIGHTON",2021
2,{BC8936BB-F58B-0E2C-E053-6C04A8C0DBF4},505000,2021-01-04,BN16 2EA,D,N,F,45,<NA>,GLENVILLE ROAD,RUSTINGTON,LITTLEHAMPTON,ARUN,WEST SUSSEX,A,A,BN,<NA>,None,Detached,Established,Freehold,"45, GLENVILLE ROAD, RUSTINGTON, LITTLEHAMPTON",2021
3,{BC8936BB-743D-0E2C-E053-6C04A8C0DBF4},430000,2021-01-04,BN2 5AD,F,N,L,"THE LEAS, 34 - 35",FLAT 1,SUSSEX SQUARE,<NA>,BRIGHTON,BRIGHTON AND HOVE,BRIGHTON AND HOVE,A,A,BN,<NA>,None,Flat/Maisonette,Established,Leasehold,"FLAT 1, THE LEAS, 34 - 35, SUSSEX SQUARE, BRIGHTON",2021
4,{BC8936BB-6FA6-0E2C-E053-6C04A8C0DBF4},800000,2021-01-04,BN20 8DL,D,N,F,25,<NA>,OLD CAMP ROAD,<NA>,EASTBOURNE,EASTBOURNE,EAST SUSSEX,A,A,BN,<NA>,None,Detached,Established,Freehold,"25, OLD CAMP ROAD, EASTBOURNE",2021


In [15]:
# === 4. Normalise schema defensively ===
def find_col(df, candidates):
    cols = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().strip()
        if key in cols:
            return cols[key]
    return None

price_col = find_col(sales, ['price', 'price_paid', 'amount'])
date_col = find_col(sales, ['date_of_transfer', 'deed_date', 'date'])
postcode_col = find_col(sales, ['postcode', 'post_code'])
ptype_col = find_col(sales, ['property_type'])
paon_col = find_col(sales, ['paon', 'primary_addressable_object_name'])
saon_col = find_col(sales, ['saon', 'secondary_addressable_object_name'])
street_col = find_col(sales, ['street'])
town_col = find_col(sales, ['town_city', 'town/city', 'town'])
district_col = find_col(sales, ['district'])
county_col = find_col(sales, ['county'])
category_col = find_col(sales, ['ppd_category_type', 'record_status'])

required_map = {'price': price_col, 'date': date_col, 'postcode': postcode_col}
missing = [k for k, v in required_map.items() if v is None]
assert not missing, f'Missing required columns after detection: {missing}'

work = pd.DataFrame()
work['price'] = pd.to_numeric(sales[price_col], errors='coerce')
work['date_of_transfer'] = pd.to_datetime(sales[date_col], errors='coerce')
work['postcode'] = sales[postcode_col].astype(str).str.upper().str.strip()
work['postcode_district'] = work['postcode'].str.extract(r'^([A-Z]{1,2}\d{1,2}[A-Z]?)', expand=False)

for out_col, src_col in {
    'property_type': ptype_col,
    'paon': paon_col,
    'saon': saon_col,
    'street': street_col,
    'town_city': town_col,
    'district': district_col,
    'county': county_col,
    'ppd_category_type': category_col,
}.items():
    work[out_col] = sales[src_col].astype(str).str.upper().str.strip() if src_col else ''

work = work.dropna(subset=['price', 'date_of_transfer', 'postcode']).copy()
work = work[work['postcode'].str.startswith(POSTCODE_PREFIX, na=False)].copy()
if STANDARD_ONLY and 'ppd_category_type' in work.columns and work['ppd_category_type'].str.len().gt(0).any():
    work = work[work['ppd_category_type'].eq('A')].copy()

if TARGET_DISTRICTS:
    work = work[work['postcode_district'].isin([d.upper() for d in TARGET_DISTRICTS])].copy()

print('Normalised rows:', len(work))
print(work[['date_of_transfer','price','postcode','postcode_district','property_type','paon','saon','street','town_city']].head())


Normalised rows: 68685
  date_of_transfer   price  postcode postcode_district property_type  \
0       2021-01-03  650000   BN3 5DL               BN3             S   
1       2021-01-04  265000   BN1 3RT               BN1             F   
2       2021-01-04  505000  BN16 2EA              BN16             D   
3       2021-01-04  430000   BN2 5AD               BN2             F   
4       2021-01-04  800000  BN20 8DL              BN20             D   

                paon               saon          street      town_city  
0                 58               <NA>   PORTLAND ROAD           HOVE  
1                  1  SECOND FLOOR FLAT  WEST HILL ROAD       BRIGHTON  
2                 45               <NA>  GLENVILLE ROAD  LITTLEHAMPTON  
3  THE LEAS, 34 - 35             FLAT 1   SUSSEX SQUARE       BRIGHTON  
4                 25               <NA>   OLD CAMP ROAD     EASTBOURNE  


In [16]:
# === 5. Build stable transaction keys and address text ===
def clean_text(x):
    if pd.isna(x):
        return ''
    s = str(x).upper().strip()
    s = re.sub(r'\s+', ' ', s)
    s = s.replace('NAN', '').strip()
    return s

def make_address(row):
    parts = [row.get('saon',''), row.get('paon',''), row.get('street',''), row.get('town_city',''), row.get('postcode','')]
    parts = [clean_text(p) for p in parts if clean_text(p)]
    return ', '.join(parts)

def make_tx_id(row):
    raw = '|'.join([
        str(row.get('date_of_transfer'))[:10],
        str(int(row.get('price'))) if pd.notna(row.get('price')) else '',
        clean_text(row.get('postcode')),
        clean_text(row.get('paon')),
        clean_text(row.get('saon')),
        clean_text(row.get('street')),
    ])
    return hashlib.sha1(raw.encode('utf-8')).hexdigest()[:16]

work['address_text'] = work.apply(make_address, axis=1)
work['transaction_id'] = work.apply(make_tx_id, axis=1)
work['sale_year'] = work['date_of_transfer'].dt.year
work['sale_month'] = work['date_of_transfer'].dt.to_period('M').astype(str)

print('Unique transaction IDs:', work['transaction_id'].nunique(), 'Rows:', len(work))
work[['transaction_id','date_of_transfer','price','postcode','address_text']].head(10)


Unique transaction IDs: 68647 Rows: 68685


,transaction_id,date_of_transfer,price,postcode,address_text
0,02007bf2c3e6052a,2021-01-03,650000,BN3 5DL,"<NA>, 58, PORTLAND ROAD, HOVE, BN3 5DL"
1,66a28a0b02bea63d,2021-01-04,265000,BN1 3RT,"SECOND FLOOR FLAT, 1, WEST HILL ROAD, BRIGHTON, BN1 3RT"
2,9c7bfaf1dccfe31b,2021-01-04,505000,BN16 2EA,"<NA>, 45, GLENVILLE ROAD, LITTLEHAMPTON, BN16 2EA"
3,629e1ea8d2250d9d,2021-01-04,430000,BN2 5AD,"FLAT 1, THE LEAS, 34 - 35, SUSSEX SQUARE, BRIGHTON, BN2 5AD"
4,10e56171ad528991,2021-01-04,800000,BN20 8DL,"<NA>, 25, OLD CAMP ROAD, EASTBOURNE, BN20 8DL"
5,50ca2244d61727ac,2021-01-04,370000,BN25 2RP,"<NA>, 4, HAWTH GROVE, SEAFORD, BN25 2RP"
6,a55ab57963fc9bb3,2021-01-04,242500,BN43 5AL,"<NA>, 43A, SOUTHDOWN ROAD, SHOREHAM-BY-SEA, BN43 5AL"
7,adcf7227aed8706d,2021-01-04,441200,BN44 3PL,"<NA>, 40, PENLANDS VALE, STEYNING, BN44 3PL"
8,374abaf6cb0589dc,2021-01-05,230000,BN1 3JB,"FLAT 12, 48 - 50, DYKE ROAD, BRIGHTON, BN1 3JB"
9,07e0938769718b9a,2021-01-05,497500,BN1 5EH,"<NA>, 102, ELDRED AVENUE, BRIGHTON, BN1 5EH"


In [17]:
# === 6. Create a balanced pilot sample ===
# Strategy: sample across postcode districts and years, so the pilot is not dominated by one town/year.

sample_pool = work.sort_values(['postcode_district','sale_year','date_of_transfer']).copy()

# Use group sampling where possible.
group_cols = ['postcode_district', 'sale_year']
num_groups = sample_pool[group_cols].drop_duplicates().shape[0]
per_group = max(1, int(np.ceil(SAMPLE_N / max(num_groups, 1))))

sample = (
    sample_pool
    .groupby(group_cols, group_keys=False)
    .apply(lambda g: g.sample(min(len(g), per_group), random_state=RANDOM_SEED))
    .sample(frac=1, random_state=RANDOM_SEED)
    .head(SAMPLE_N)
    .reset_index(drop=True)
)

print('Sample rows:', len(sample))
print('Districts:', sorted(sample['postcode_district'].dropna().unique())[:30], '...')
sample[['transaction_id','sale_year','date_of_transfer','price','postcode_district','postcode','address_text']].head(20)


Sample rows: 250
Districts: ['BN1', 'BN10', 'BN11', 'BN12', 'BN13', 'BN14', 'BN15', 'BN16', 'BN17', 'BN18', 'BN2', 'BN20', 'BN21', 'BN22', 'BN23', 'BN24', 'BN25', 'BN26', 'BN27', 'BN3', 'BN41', 'BN42', 'BN43', 'BN44', 'BN45', 'BN5', 'BN6', 'BN7', 'BN8', 'BN9'] ...


/tmp/ipykernel_21125/2986407135.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), per_group), random_state=RANDOM_SEED))


,transaction_id,sale_year,date_of_transfer,price,postcode_district,postcode,address_text
0,6662cdb569412bc4,2022,2022-08-10,235000,BN42,BN42 4FL,"<NA>, 116, SOUTHWICK SQUARE, BRIGHTON, BN42 4FL"
1,8039747ddacbb7fe,2022,2022-05-20,380000,BN16,BN16 3QL,"<NA>, 61, OLD MANOR ROAD, LITTLEHAMPTON, BN16 3QL"
2,b662b40723258915,2025,2025-06-04,445000,BN43,BN43 5PS,"<NA>, 2, SUSSEX WHARF, SHOREHAM-BY-SEA, BN43 5PS"
3,46d73c55fcd7ab29,2022,2022-08-31,455000,BN42,BN42 4WG,"<NA>, 27, CROMLEIGH WAY, BRIGHTON, BN42 4WG"
4,2ae5e8296e2bcede,2024,2024-02-09,330000,BN22,BN22 7HN,"<NA>, 6, RYLSTONE ROAD, EASTBOURNE, BN22 7HN"
5,81f3186e48d1c077,2022,2022-03-15,260000,BN43,BN43 6WN,"FLAT 37, ROSSLYN COURT, ROSSLYN ROAD, SHOREHAM-BY-SEA, BN43 6WN"
6,2320e7e0dd78d5a8,2024,2024-05-24,220000,BN15,BN15 8AU,"<NA>, 156A, SOUTH STREET, LANCING, BN15 8AU"
7,55c07e9feebe6c73,2022,2022-12-09,475000,BN5,BN5 9YF,"<NA>, BEECHWOOD, WOOD LANE, HENFIELD, BN5 9YF"
8,873c10bf2db112b8,2025,2025-12-03,735000,BN45,BN45 7FQ,"<NA>, RIDGEMOUNT, SCHOOL LANE, BRIGHTON, BN45 7FQ"
9,670c3a3056888bc6,2025,2025-03-04,630000,BN16,BN16 2AT,"<NA>, 1, HAWKE CLOSE, LITTLEHAMPTON, BN16 2AT"


In [18]:
# === 7. Generate search/evidence URLs for manual review ===
# These URLs are for human review. They help discover whether public evidence still exists.
# We store candidate evidence separately and never overwrite Land Registry facts.

SEARCH_ENGINE = 'https://www.google.com/search?q='
# Alternatives you can use manually if Google blocks automated notebook browsing:
# SEARCH_ENGINE = 'https://www.bing.com/search?q='
# SEARCH_ENGINE = 'https://duckduckgo.com/?q='

PORTAL_TERMS = ['Rightmove', 'Zoopla', 'OnTheMarket']

def quote_query(q):
    return SEARCH_ENGINE + urllib.parse.quote_plus(q)

def build_queries(row):
    addr = row['address_text']
    pc = row['postcode']
    price = int(row['price']) if pd.notna(row['price']) else ''
    year = int(row['sale_year']) if pd.notna(row['sale_year']) else ''
    # Keep queries specific but not too brittle.
    q1 = f'"{pc}" "{price}" sold estate agent'
    q2 = f'"{addr}" estate agent sold'
    q3 = f'"{pc}" "Rightmove" "sold" "{year}"'
    q4 = f'"{pc}" "Zoopla" "sold" "{year}"'
    return [q1, q2, q3, q4]

rows = []
for _, r in sample.iterrows():
    queries = build_queries(r)
    rows.append({
        'transaction_id': r['transaction_id'],
        'sale_date': r['date_of_transfer'].date().isoformat(),
        'sale_year': int(r['sale_year']),
        'price': int(r['price']),
        'postcode': r['postcode'],
        'postcode_district': r['postcode_district'],
        'address_text': r['address_text'],
        'query_1': queries[0],
        'search_url_1': quote_query(queries[0]),
        'query_2': queries[1],
        'search_url_2': quote_query(queries[1]),
        'query_3': queries[2],
        'search_url_3': quote_query(queries[2]),
        'query_4': queries[3],
        'search_url_4': quote_query(queries[3]),
        'review_status': 'pending',
        'candidate_agent': '',
        'candidate_source_url': '',
        'evidence_note': '',
        'confidence': '',
    })

tasks = pd.DataFrame(rows)
tasks_path = REPORTS / f'agent_review_tasks_BN_last5y_{RUN_UTC}.csv'
tasks.to_csv(tasks_path, index=False)

print('Saved review task CSV:', tasks_path)
tasks.head()


Saved review task CSV: /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agent_review_tasks_BN_last5y_20260427T132744Z.csv


,transaction_id,sale_date,sale_year,price,postcode,postcode_district,address_text,query_1,search_url_1,query_2,search_url_2,query_3,search_url_3,query_4,search_url_4,review_status,candidate_agent,candidate_source_url,evidence_note,confidence
0,6662cdb569412bc4,2022-08-10,2022,235000,BN42 4FL,BN42,"<NA>, 116, SOUTHWICK SQUARE, BRIGHTON, BN42 4FL","""BN42 4FL"" ""235000"" sold estate agent",https://www.google.com/search?q=%22BN42+4FL%22+%22235000%22+sold+estate+agent,"""<NA>, 116, SOUTHWICK SQUARE, BRIGHTON, BN42 4FL"" estate agent sold",https://www.google.com/search?q=%22%3CNA%3E%2C+116%2C+SOUTHWICK+SQUARE%2C+BRIGHTON%2C+BN42+4FL%22+estate+agent+sold,"""BN42 4FL"" ""Rightmove"" ""sold"" ""2022""",https://www.google.com/search?q=%22BN42+4FL%22+%22Rightmove%22+%22sold%22+%222022%22,"""BN42 4FL"" ""Zoopla"" ""sold"" ""2022""",https://www.google.com/search?q=%22BN42+4FL%22+%22Zoopla%22+%22sold%22+%222022%22,pending,,,,
1,8039747ddacbb7fe,2022-05-20,2022,380000,BN16 3QL,BN16,"<NA>, 61, OLD MANOR ROAD, LITTLEHAMPTON, BN16 3QL","""BN16 3QL"" ""380000"" sold estate agent",https://www.google.com/search?q=%22BN16+3QL%22+%22380000%22+sold+estate+agent,"""<NA>, 61, OLD MANOR ROAD, LITTLEHAMPTON, BN16 3QL"" estate agent sold",https://www.google.com/search?q=%22%3CNA%3E%2C+61%2C+OLD+MANOR+ROAD%2C+LITTLEHAMPTON%2C+BN16+3QL%22+estate+agent+sold,"""BN16 3QL"" ""Rightmove"" ""sold"" ""2022""",https://www.google.com/search?q=%22BN16+3QL%22+%22Rightmove%22+%22sold%22+%222022%22,"""BN16 3QL"" ""Zoopla"" ""sold"" ""2022""",https://www.google.com/search?q=%22BN16+3QL%22+%22Zoopla%22+%22sold%22+%222022%22,pending,,,,
2,b662b40723258915,2025-06-04,2025,445000,BN43 5PS,BN43,"<NA>, 2, SUSSEX WHARF, SHOREHAM-BY-SEA, BN43 5PS","""BN43 5PS"" ""445000"" sold estate agent",https://www.google.com/search?q=%22BN43+5PS%22+%22445000%22+sold+estate+agent,"""<NA>, 2, SUSSEX WHARF, SHOREHAM-BY-SEA, BN43 5PS"" estate agent sold",https://www.google.com/search?q=%22%3CNA%3E%2C+2%2C+SUSSEX+WHARF%2C+SHOREHAM-BY-SEA%2C+BN43+5PS%22+estate+agent+sold,"""BN43 5PS"" ""Rightmove"" ""sold"" ""2025""",https://www.google.com/search?q=%22BN43+5PS%22+%22Rightmove%22+%22sold%22+%222025%22,"""BN43 5PS"" ""Zoopla"" ""sold"" ""2025""",https://www.google.com/search?q=%22BN43+5PS%22+%22Zoopla%22+%22sold%22+%222025%22,pending,,,,
3,46d73c55fcd7ab29,2022-08-31,2022,455000,BN42 4WG,BN42,"<NA>, 27, CROMLEIGH WAY, BRIGHTON, BN42 4WG","""BN42 4WG"" ""455000"" sold estate agent",https://www.google.com/search?q=%22BN42+4WG%22+%22455000%22+sold+estate+agent,"""<NA>, 27, CROMLEIGH WAY, BRIGHTON, BN42 4WG"" estate agent sold",https://www.google.com/search?q=%22%3CNA%3E%2C+27%2C+CROMLEIGH+WAY%2C+BRIGHTON%2C+BN42+4WG%22+estate+agent+sold,"""BN42 4WG"" ""Rightmove"" ""sold"" ""2022""",https://www.google.com/search?q=%22BN42+4WG%22+%22Rightmove%22+%22sold%22+%222022%22,"""BN42 4WG"" ""Zoopla"" ""sold"" ""2022""",https://www.google.com/search?q=%22BN42+4WG%22+%22Zoopla%22+%22sold%22+%222022%22,pending,,,,
4,2ae5e8296e2bcede,2024-02-09,2024,330000,BN22 7HN,BN22,"<NA>, 6, RYLSTONE ROAD, EASTBOURNE, BN22 7HN","""BN22 7HN"" ""330000"" sold estate agent",https://www.google.com/search?q=%22BN22+7HN%22+%22330000%22+sold+estate+agent,"""<NA>, 6, RYLSTONE ROAD, EASTBOURNE, BN22 7HN"" estate agent sold",https://www.google.com/search?q=%22%3CNA%3E%2C+6%2C+RYLSTONE+ROAD%2C+EASTBOURNE%2C+BN22+7HN%22+estate+agent+sold,"""BN22 7HN"" ""Rightmove"" ""sold"" ""2024""",https://www.google.com/search?q=%22BN22+7HN%22+%22Rightmove%22+%22sold%22+%222024%22,"""BN22 7HN"" ""Zoopla"" ""sold"" ""2024""",https://www.google.com/search?q=%22BN22+7HN%22+%22Zoopla%22+%22sold%22+%222024%22,pending,,,,


In [19]:
# === 8. Optional: create a smaller human-friendly review sheet ===
# In Colab, download or open this CSV from Drive. Fill candidate_agent/source/confidence fields manually.
# Confidence values recommended: high, medium, low, unknown.

manual_cols = [
    'transaction_id','sale_date','price','postcode','address_text',
    'search_url_1','search_url_2','search_url_3','search_url_4',
    'review_status','candidate_agent','candidate_source_url','evidence_note','confidence'
]
manual_sheet = tasks[manual_cols].copy()
manual_path = REPORTS / f'agent_manual_review_sheet_BN_last5y_{RUN_UTC}.csv'
manual_sheet.to_csv(manual_path, index=False)

print('Manual review sheet saved:', manual_path)
manual_sheet.head(10)


Manual review sheet saved: /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agent_manual_review_sheet_BN_last5y_20260427T132744Z.csv


,transaction_id,sale_date,price,postcode,address_text,search_url_1,search_url_2,search_url_3,search_url_4,review_status,candidate_agent,candidate_source_url,evidence_note,confidence
0,6662cdb569412bc4,2022-08-10,235000,BN42 4FL,"<NA>, 116, SOUTHWICK SQUARE, BRIGHTON, BN42 4FL",https://www.google.com/search?q=%22BN42+4FL%22+%22235000%22+sold+estate+agent,https://www.google.com/search?q=%22%3CNA%3E%2C+116%2C+SOUTHWICK+SQUARE%2C+BRIGHTON%2C+BN42+4FL%22+estate+agent+sold,https://www.google.com/search?q=%22BN42+4FL%22+%22Rightmove%22+%22sold%22+%222022%22,https://www.google.com/search?q=%22BN42+4FL%22+%22Zoopla%22+%22sold%22+%222022%22,pending,,,,
1,8039747ddacbb7fe,2022-05-20,380000,BN16 3QL,"<NA>, 61, OLD MANOR ROAD, LITTLEHAMPTON, BN16 3QL",https://www.google.com/search?q=%22BN16+3QL%22+%22380000%22+sold+estate+agent,https://www.google.com/search?q=%22%3CNA%3E%2C+61%2C+OLD+MANOR+ROAD%2C+LITTLEHAMPTON%2C+BN16+3QL%22+estate+agent+sold,https://www.google.com/search?q=%22BN16+3QL%22+%22Rightmove%22+%22sold%22+%222022%22,https://www.google.com/search?q=%22BN16+3QL%22+%22Zoopla%22+%22sold%22+%222022%22,pending,,,,
2,b662b40723258915,2025-06-04,445000,BN43 5PS,"<NA>, 2, SUSSEX WHARF, SHOREHAM-BY-SEA, BN43 5PS",https://www.google.com/search?q=%22BN43+5PS%22+%22445000%22+sold+estate+agent,https://www.google.com/search?q=%22%3CNA%3E%2C+2%2C+SUSSEX+WHARF%2C+SHOREHAM-BY-SEA%2C+BN43+5PS%22+estate+agent+sold,https://www.google.com/search?q=%22BN43+5PS%22+%22Rightmove%22+%22sold%22+%222025%22,https://www.google.com/search?q=%22BN43+5PS%22+%22Zoopla%22+%22sold%22+%222025%22,pending,,,,
3,46d73c55fcd7ab29,2022-08-31,455000,BN42 4WG,"<NA>, 27, CROMLEIGH WAY, BRIGHTON, BN42 4WG",https://www.google.com/search?q=%22BN42+4WG%22+%22455000%22+sold+estate+agent,https://www.google.com/search?q=%22%3CNA%3E%2C+27%2C+CROMLEIGH+WAY%2C+BRIGHTON%2C+BN42+4WG%22+estate+agent+sold,https://www.google.com/search?q=%22BN42+4WG%22+%22Rightmove%22+%22sold%22+%222022%22,https://www.google.com/search?q=%22BN42+4WG%22+%22Zoopla%22+%22sold%22+%222022%22,pending,,,,
4,2ae5e8296e2bcede,2024-02-09,330000,BN22 7HN,"<NA>, 6, RYLSTONE ROAD, EASTBOURNE, BN22 7HN",https://www.google.com/search?q=%22BN22+7HN%22+%22330000%22+sold+estate+agent,https://www.google.com/search?q=%22%3CNA%3E%2C+6%2C+RYLSTONE+ROAD%2C+EASTBOURNE%2C+BN22+7HN%22+estate+agent+sold,https://www.google.com/search?q=%22BN22+7HN%22+%22Rightmove%22+%22sold%22+%222024%22,https://www.google.com/search?q=%22BN22+7HN%22+%22Zoopla%22+%22sold%22+%222024%22,pending,,,,
5,81f3186e48d1c077,2022-03-15,260000,BN43 6WN,"FLAT 37, ROSSLYN COURT, ROSSLYN ROAD, SHOREHAM-BY-SEA, BN43 6WN",https://www.google.com/search?q=%22BN43+6WN%22+%22260000%22+sold+estate+agent,https://www.google.com/search?q=%22FLAT+37%2C+ROSSLYN+COURT%2C+ROSSLYN+ROAD%2C+SHOREHAM-BY-SEA%2C+BN43+6WN%22+estate+agent+sold,https://www.google.com/search?q=%22BN43+6WN%22+%22Rightmove%22+%22sold%22+%222022%22,https://www.google.com/search?q=%22BN43+6WN%22+%22Zoopla%22+%22sold%22+%222022%22,pending,,,,
6,2320e7e0dd78d5a8,2024-05-24,220000,BN15 8AU,"<NA>, 156A, SOUTH STREET, LANCING, BN15 8AU",https://www.google.com/search?q=%22BN15+8AU%22+%22220000%22+sold+estate+agent,https://www.google.com/search?q=%22%3CNA%3E%2C+156A%2C+SOUTH+STREET%2C+LANCING%2C+BN15+8AU%22+estate+agent+sold,https://www.google.com/search?q=%22BN15+8AU%22+%22Rightmove%22+%22sold%22+%222024%22,https://www.google.com/search?q=%22BN15+8AU%22+%22Zoopla%22+%22sold%22+%222024%22,pending,,,,
7,55c07e9feebe6c73,2022-12-09,475000,BN5 9YF,"<NA>, BEECHWOOD, WOOD LANE, HENFIELD, BN5 9YF",https://www.google.com/search?q=%22BN5+9YF%22+%22475000%22+sold+estate+agent,https://www.google.com/search?q=%22%3CNA%3E%2C+BEECHWOOD%2C+WOOD+LANE%2C+HENFIELD%2C+BN5+9YF%22+estate+agent+sold,https://www.google.com/search?q=%22BN5+9YF%22+%22Rightmove%22+%22sold%22+%222022%22,https://www.google.com/search?q=%22BN5+9YF%22+%22Zoopla%22+%22sold%22+%222022%22,pending,,,,
8,873c10bf2db112b8,2025-12-03,735000,BN45 7FQ,"<NA>, RIDGEMOUN

In [20]:
# === 9. Optional evidence parser for manually saved HTML/text files ===
# Workflow:
# 1. Open search URLs manually.
# 2. If a page clearly shows the sale/listing and agent, save the page HTML or copy text into:
#    /content/drive/MyDrive/Addresses26_Data/evidence/agent_attribution/<transaction_id>.txt
# 3. Re-run this section to extract likely agent phrases.
#
# This parser is intentionally simple. It gives candidates for human checking, not final truth.

AGENT_PATTERNS = [
    r'(?i)marketed by[:\s]+([A-Za-z0-9 &.,\-]+)',
    r'(?i)estate agent[s]?[:\s]+([A-Za-z0-9 &.,\-]+)',
    r'(?i)sold by[:\s]+([A-Za-z0-9 &.,\-]+)',
    r'(?i)listed by[:\s]+([A-Za-z0-9 &.,\-]+)',
]

def parse_agent_candidates(text):
    candidates = []
    if not isinstance(text, str):
        return candidates
    compact = re.sub(r'\s+', ' ', text)
    for pat in AGENT_PATTERNS:
        for m in re.finditer(pat, compact):
            cand = m.group(1).strip(' -:|,.;')
            if 3 <= len(cand) <= 80:
                candidates.append(cand)
    # de-duplicate preserving order
    seen = set()
    out = []
    for c in candidates:
        key = c.lower()
        if key not in seen:
            seen.add(key)
            out.append(c)
    return out[:5]

parsed_rows = []
for txt_path in sorted(EVIDENCE.glob('*.txt')) + sorted(EVIDENCE.glob('*.html')):
    txid = txt_path.stem.split('__')[0]
    try:
        text = txt_path.read_text(encoding='utf-8', errors='ignore')
    except Exception as e:
        parsed_rows.append({'transaction_id': txid, 'evidence_file': str(txt_path), 'error': str(e)})
        continue
    candidates = parse_agent_candidates(text)
    parsed_rows.append({
        'transaction_id': txid,
        'evidence_file': str(txt_path),
        'candidate_agents_parsed': ' | '.join(candidates),
        'error': ''
    })

parsed = pd.DataFrame(parsed_rows)
parsed_path = REPORTS / f'agent_evidence_parsed_{RUN_UTC}.csv'
parsed.to_csv(parsed_path, index=False)
print('Parsed evidence files:', len(parsed))
print('Saved:', parsed_path)
parsed.head(20)


Parsed evidence files: 0
Saved: /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agent_evidence_parsed_20260427T132744Z.csv


""


In [21]:
# === 10. Scoring framework for manually reviewed attributions ===
# After you fill a review sheet, put it in REPORTS and set REVIEWED_CSV below.
# The notebook will turn it into a structured candidate attribution table.

REVIEWED_CSV = None  # e.g. REPORTS / 'agent_manual_review_sheet_BN_last5y_REVIEWED.csv'

CONFIDENCE_SCORE = {
    'high': 0.90,
    'medium': 0.60,
    'low': 0.30,
    'unknown': 0.0,
    '': 0.0,
}

if REVIEWED_CSV is not None and Path(REVIEWED_CSV).exists():
    reviewed = pd.read_csv(REVIEWED_CSV)
else:
    reviewed = manual_sheet.copy()

for col in ['candidate_agent','candidate_source_url','evidence_note','confidence','review_status']:
    if col not in reviewed.columns:
        reviewed[col] = ''

reviewed['confidence_norm'] = reviewed['confidence'].fillna('').astype(str).str.lower().str.strip()
reviewed['confidence_score'] = reviewed['confidence_norm'].map(CONFIDENCE_SCORE).fillna(0.0)
reviewed['checked_utc'] = datetime.now(timezone.utc).isoformat()
reviewed['attribution_method'] = 'manual_public_evidence_review'
reviewed['is_agent_confirmed'] = reviewed['confidence_score'].ge(0.85)

attrib_cols = [
    'transaction_id','sale_date','price','postcode','address_text',
    'candidate_agent','candidate_source_url','confidence','confidence_score',
    'is_agent_confirmed','evidence_note','review_status','attribution_method','checked_utc'
]
attrib = reviewed[attrib_cols].copy()

attrib_path = REPORTS / f'agent_attribution_candidates_{RUN_UTC}.csv'
attrib.to_csv(attrib_path, index=False)
print('Saved attribution candidate table:', attrib_path)
attrib.head()


Saved attribution candidate table: /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agent_attribution_candidates_20260427T132744Z.csv


,transaction_id,sale_date,price,postcode,address_text,candidate_agent,candidate_source_url,confidence,confidence_score,is_agent_confirmed,evidence_note,review_status,attribution_method,checked_utc
0,6662cdb569412bc4,2022-08-10,235000,BN42 4FL,"<NA>, 116, SOUTHWICK SQUARE, BRIGHTON, BN42 4FL",,,,0.0,False,,pending,manual_public_evidence_review,2026-04-27T13:27:50.961199+00:00
1,8039747ddacbb7fe,2022-05-20,380000,BN16 3QL,"<NA>, 61, OLD MANOR ROAD, LITTLEHAMPTON, BN16 3QL",,,,0.0,False,,pending,manual_public_evidence_review,2026-04-27T13:27:50.961199+00:00
2,b662b40723258915,2025-06-04,445000,BN43 5PS,"<NA>, 2, SUSSEX WHARF, SHOREHAM-BY-SEA, BN43 5PS",,,,0.0,False,,pending,manual_public_evidence_review,2026-04-27T13:27:50.961199+00:00
3,46d73c55fcd7ab29,2022-08-31,455000,BN42 4WG,"<NA>, 27, CROMLEIGH WAY, BRIGHTON, BN42 4WG",,,,0.0,False,,pending,manual_public_evidence_review,2026-04-27T13:27:50.961199+00:00
4,2ae5e8296e2bcede,2024-02-09,330000,BN22 7HN,"<NA>, 6, RYLSTONE ROAD, EASTBOURNE, BN22 7HN",,,,0.0,False,,pending,manual_public_evidence_review,2026-04-27T13:27:50.961199+00:00


In [22]:
# === 11. Pilot analytics ===
summary = {
    'run_utc': RUN_UTC,
    'years': YEARS,
    'postcode_prefix': POSTCODE_PREFIX,
    'standard_only': STANDARD_ONLY,
    'rows_loaded_normalised': int(len(work)),
    'sample_rows': int(len(sample)),
    'districts_in_sample': int(sample['postcode_district'].nunique()),
    'task_csv': str(tasks_path),
    'manual_review_csv': str(manual_path),
    'attribution_candidates_csv': str(attrib_path),
    'notes': [
        'Land Registry data is the factual core; agent attribution is an enrichment layer.',
        'No portal scraping or anti-bot bypassing is performed by this notebook.',
        'Agent evidence should be manually reviewed and confidence-scored before analysis.'
    ]
}

summary_path = REPORTS / f'agent_pilot_manifest_{RUN_UTC}.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
print('Saved manifest:', summary_path)


{
  "run_utc": "20260427T132744Z",
  "years": [
    2021,
    2022,
    2023,
    2024,
    2025
  ],
  "postcode_prefix": "BN",
  "standard_only": true,
  "rows_loaded_normalised": 68685,
  "sample_rows": 250,
  "districts_in_sample": 30,
  "task_csv": "/content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agent_review_tasks_BN_last5y_20260427T132744Z.csv",
  "manual_review_csv": "/content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agent_manual_review_sheet_BN_last5y_20260427T132744Z.csv",
  "attribution_candidates_csv": "/content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agent_attribution_candidates_20260427T132744Z.csv",
  "notes": [
    "Land Registry data is the factual core; agent attribution is an enrichment layer.",
    "No portal scraping or anti-bot bypassing is performed by this notebook.",
    "Agent evidence should be manually reviewed and confidence-scored before analysis."
  ]
}
Saved manifest: /content/drive/MyDrive/Addresses26_D

In [23]:
# === 12. Basic counts for deciding whether to scale ===
print('Sample by year:')
display(sample.groupby('sale_year').size().rename('sample_rows').reset_index())

print('Sample by postcode district:')
display(sample.groupby('postcode_district').size().rename('sample_rows').reset_index().sort_values('sample_rows', ascending=False).head(30))

print('Attribution status from current/manual sheet:')
display(attrib.groupby(['review_status','confidence'], dropna=False).size().rename('rows').reset_index())


Sample by year:


,sale_year,sample_rows
0,2021,50
1,2022,49
2,2023,54
3,2024,45
4,2025,52


Sample by postcode district:


,postcode_district,sample_rows
0,BN1,10
3,BN12,10
13,BN22,10
12,BN21,10
6,BN15,10
27,BN7,10
24,BN45,10
26,BN6,9
9,BN18,9
22,BN43,9


Attribution status from current/manual sheet:


,review_status,confidence,rows
0,pending,,250


## Recommended pilot loop

1. Open `agent_manual_review_sheet_...csv` from Drive.
2. Pick 25–50 rows first, not all 250.
3. Use the generated search URLs manually.
4. Fill `candidate_agent`, `candidate_source_url`, `evidence_note`, and `confidence`.
5. Save the reviewed CSV back into `reports/agent_attribution/`.
6. Set `REVIEWED_CSV` in section 10 and re-run sections 10–12.
7. Analyse the hit rate. If fewer than ~30% of rows can be confidently attributed, change strategy before scaling.

Keep this separate from the verified Price Paid dataset. The agent table is evidence-based enrichment, not official Land Registry data.
